# SpaceX SQL Data Analysis Exercise

This notebook completes the ten SQL tasks from the provided assignment. It loads the SpaceX launch dataset into an in-memory SQLite database, removes rows without a date, runs each query, and displays the result.

The dataset is provided by the IBM Skills Network course and is downloaded when the notebook runs. SQL table and column identifiers are quoted where needed because the source column names contain mixed case and punctuation.


In [1]:
import sqlite3
import pandas as pd
from IPython.display import display

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
df = pd.read_csv(DATA_URL)
df = df.dropna(subset=["Date"])
conn = sqlite3.connect(":memory:")
df.to_sql("SPACEXTABLE", conn, if_exists="replace", index=False)
print(f"Loaded {len(df)} dated launch records into SPACEXTABLE.")


Loaded 101 dated launch records into SPACEXTABLE.


## Tasks and solutions


### Task 1 — Unique launch sites


In [2]:
query = """SELECT DISTINCT "Launch_Site" FROM SPACEXTABLE ORDER BY "Launch_Site";"""
display(pd.read_sql_query(query, conn))


Launch_Site
-----------
CCAFS LC-40
CCAFS SLC-40
KSC LC-39A
VAFB SLC-4E

### Task 2 — First five CCA launch site records


In [3]:
query = """SELECT * FROM SPACEXTABLE WHERE "Launch_Site" LIKE 'CCA%' ORDER BY "Date" LIMIT 5;"""
display(pd.read_sql_query(query, conn))


Date | Time (UTC) | Booster_Version | Launch_Site | Payload | PAYLOAD_MASS__KG_ | Orbit | Customer | Mission_Outcome | Landing_Outcome
-----+------------+-----------------+-------------+---------+-------------------+-------+----------+-----------------+----------------
2010-06-04 | 18:45:00 | F9 v1.0  B0003 | CCAFS LC-40 | Dragon Spacecraft Qualification Unit | 0 | LEO | SpaceX | Success | Failure (parachute)
2010-12-08 | 15:43:00 | F9 v1.0  B0004 | CCAFS LC-40 | Dragon demo flight C1, two CubeSats, barrel of Brouere cheese | 0 | LEO (ISS) | NASA (COTS) NRO | Success | Failure (parachute)
2012-05-22 | 7:44:00 | F9 v1.0  B0005 | CCAFS LC-40 | Dragon demo flight C2 | 525 | LEO (ISS) | NASA (COTS) | Success | No attempt
2012-10-08 | 0:35:00 | F9 v1.0  B0006 | CCAFS LC-40 | SpaceX CRS-1 | 500 | LEO (ISS) | NASA (CRS) | Success | No attempt
2013-03-01 | 15:10:00 | F9 v1.0  B0007 | CCAFS LC-40 | SpaceX CRS-2 | 677 | LEO (ISS) | NASA (CRS) | Success | No attempt

### Task 3 — Payload mass carried for NASA (CRS)


In [4]:
query = """SELECT SUM(CAST("PAYLOAD_MASS__KG_" AS REAL)) AS total_payload_mass_kg FROM SPACEXTABLE WHERE "Customer" LIKE '%NASA (CRS)%';"""
display(pd.read_sql_query(query, conn))


total_payload_mass_kg
---------------------
48213.0

### Task 4 — Average payload mass for F9 v1.1


In [5]:
query = """SELECT AVG(CAST("PAYLOAD_MASS__KG_" AS REAL)) AS average_payload_mass_kg FROM SPACEXTABLE WHERE "Booster_Version" LIKE 'F9 v1.1%';"""
display(pd.read_sql_query(query, conn))


average_payload_mass_kg
-----------------------
2534.6666666666665

### Task 5 — First successful ground pad landing


In [6]:
query = """SELECT MIN("Date") AS first_successful_ground_pad_landing FROM SPACEXTABLE WHERE "Landing_Outcome" = 'Success (ground pad)';"""
display(pd.read_sql_query(query, conn))


first_successful_ground_pad_landing
-----------------------------------
2015-12-22

### Task 6 — Drone ship successes with payload between 4,000 and 6,000 kg


In [7]:
query = """SELECT "Booster_Version", CAST("PAYLOAD_MASS__KG_" AS REAL) AS payload_mass_kg FROM SPACEXTABLE WHERE "Landing_Outcome" = 'Success (drone ship)' AND CAST("PAYLOAD_MASS__KG_" AS REAL) > 4000 AND CAST("PAYLOAD_MASS__KG_" AS REAL) < 6000 ORDER BY "Booster_Version";"""
display(pd.read_sql_query(query, conn))


Booster_Version | payload_mass_kg
----------------+----------------
F9 FT  B1021.2 | 5300.0
F9 FT  B1031.2 | 5200.0
F9 FT B1022 | 4696.0
F9 FT B1026 | 4600.0

### Task 7 — Successful and failed mission outcome counts


In [8]:
query = """SELECT TRIM("Mission_Outcome") AS mission_outcome, COUNT(*) AS mission_count FROM SPACEXTABLE GROUP BY TRIM("Mission_Outcome") ORDER BY mission_outcome;"""
display(pd.read_sql_query(query, conn))


mission_outcome | mission_count
----------------+--------------
Failure (in flight) | 1
Success | 99
Success (payload status unclear) | 1

### Task 8 — Booster versions carrying the maximum payload


In [9]:
query = """SELECT "Booster_Version", CAST("PAYLOAD_MASS__KG_" AS REAL) AS payload_mass_kg FROM SPACEXTABLE WHERE CAST("PAYLOAD_MASS__KG_" AS REAL) = (SELECT MAX(CAST("PAYLOAD_MASS__KG_" AS REAL)) FROM SPACEXTABLE);"""
display(pd.read_sql_query(query, conn))


Booster_Version | payload_mass_kg
----------------+----------------
F9 B5 B1048.4 | 15600.0
F9 B5 B1049.4 | 15600.0
F9 B5 B1051.3 | 15600.0
F9 B5 B1056.4 | 15600.0
F9 B5 B1048.5 | 15600.0
F9 B5 B1051.4 | 15600.0
F9 B5 B1049.5 | 15600.0
F9 B5 B1060.2  | 15600.0
F9 B5 B1058.3  | 15600.0
F9 B5 B1051.6 | 15600.0
F9 B5 B1060.3 | 15600.0
F9 B5 B1049.7  | 15600.0

### Task 9 — 2015 drone ship landing failures


In [10]:
query = """SELECT SUBSTR("Date", 6, 2) AS month, "Landing_Outcome", "Booster_Version", "Launch_Site" FROM SPACEXTABLE WHERE SUBSTR("Date", 1, 4) = '2015' AND "Landing_Outcome" = 'Failure (drone ship)' ORDER BY "Date";"""
display(pd.read_sql_query(query, conn))


month | Landing_Outcome | Booster_Version | Launch_Site
------+-----------------+-----------------+------------
01 | Failure (drone ship) | F9 v1.1 B1012 | CCAFS LC-40
04 | Failure (drone ship) | F9 v1.1 B1015 | CCAFS LC-40

### Task 10 — Landing outcome counts, ranked by frequency


In [11]:
query = """SELECT "Landing_Outcome", COUNT(*) AS outcome_count FROM SPACEXTABLE WHERE "Date" BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY "Landing_Outcome" ORDER BY outcome_count DESC, "Landing_Outcome";"""
display(pd.read_sql_query(query, conn))


Landing_Outcome | outcome_count
----------------+--------------
No attempt | 10
Failure (drone ship) | 5
Success (drone ship) | 5
Controlled (ocean) | 3
Success (ground pad) | 3
Failure (parachute) | 2
Uncontrolled (ocean) | 2
Precluded (drone ship) | 1